# Logistic regression: the Challenger O-rings

On the night of 27 January 1986, engineers at Morton Thiokol and NASA debated by teleconference whether the space shuttle Challenger should be launched the next morning at a forecast 31°F, far colder than any earlier launch. They looked at a plot of the O-ring damage of past flights against temperature, saw no pattern, and launched; the rocket's O-rings failed and the seven astronauts died. Dalal, Fowlkes and Hoadley (1989) showed what a logistic regression on all the flights would have said that night: a strong temperature effect, about five of the six field-joint O-rings expected to be damaged at 31°F, and at least a 13% chance of a catastrophic failure, against 2% had the launch waited for 60°F. Their models, diagnostics and risk assessment are replicated here, value by value.

The data is `challenger`, from PollisDatasets: Table 1 of the paper, the O-ring damage of the 23 flights whose rocket motors were recovered, with the temperature and leak-check pressures of each. The models are fitted with GLM.jl; the bootstrap, the influence statistics and the risk chain are a few lines each.

In [ ]:
Pollis.packages("GLM", "Distributions", "StatsBase", "PollisDatasets", "Plots")

## Set-up

The flights as GLM.jl needs them: `flights`, with the share of the six field-joint primary O-rings that suffered erosion or blowby, `distressed`, and whether any did, `incident`; `six`, the six joints of each flight, as binomial weights; `new`, the conditions at which a model predicts; and `replicate`, which prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using GLM, Distributions, StatsBase, PollisDatasets, Plots, Printf, Random, Statistics

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-30s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, published)
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-30s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

ch = dataset("challenger");  # one row per flight, Table 1 (p. 949)
hasproperty(ch, :field_blowby) || error("This workflow needs the 23-flight Table 1 version of the challenger dataset: update PollisDatasets")
flights = (temperature = Float64.(ch.temperature),  # joint temperature at launch, °F
	pressure = Float64.(ch.pressure),  # field-joint leak-check pressure, psi
	distressed = ch.n_failures ./ 6,  # share of the six field-joint primary O-rings with erosion or blowby
	incident = Float64.(ch.n_failures .> 0),  # any of them
	eroded = ch.field_erosion ./ 6);  # share with primary erosion (Section 8)
six = fill(6.0, 23);  # six field joints per flight: the binomial n
new(t; pressure=200) = (temperature = Float64.(collect(t)), pressure = fill(Float64(pressure), length(t)));  # Challenger: 200 psi

## 1. The plot of the teleconference

The plot discussed that night showed only the flights with O-ring damage (Fig. 1a, p. 946): damage at 53°F and at 75°F, a U shape, and no evidence of a temperature effect. The Rogers Commission named this a mistake (p. 945): the 16 flights without damage were left off, though they say the most. With them (Fig. 1b), every flight below 65°F had damage, and only 3 of the 19 above.

In [ ]:
damaged = ch.n_failures .> 0
cold = ch.temperature .< 65
@printf("below 65°F: %d of %d flights damaged; at 65°F or above: %d of %d\n", sum(damaged .& cold), sum(cold), sum(damaged .& .!cold), sum(.!cold))
panel(rows, title) = scatter(ch.temperature[rows], ch.n_failures[rows], title=title, xlims=(50, 82), ylims=(-0.3, 3), yticks=0:3,
	series_annotations=text.(ch.flight[rows], 7, :bottom), markersize=4, label="")
plot(panel(damaged, "a. the flights with damage"), panel(trues(23), "b. all 23 flights"), layout=(1, 2), size=(820, 330),
	xlabel="joint temperature, °F", ylabel="damaged O-rings")

## 2. The binomial model

Each flight has six field-joint primary O-rings, and the number X damaged is taken as binomial, n = 6, with a probability p(t, s) per joint that depends on the temperature t and the leak-check pressure s through a logistic regression (p. 948):

log[p / (1 − p)] = α + βt + γs.   (3.1)

GLM.jl fits it as a binomial GLM on the share damaged, with six as the weight. Its deviance is the paper's lack-of-fit statistic G². Dropping the pressure gives Model (3.2), and the increase in G², chi-square with 1 df under the hypothesis of no pressure effect, decides whether pressure stays: if `pressure` is false, the rest of the notebook uses Model (3.2), as the paper does.

In [ ]:
f31 = @formula(distressed ~ temperature + pressure)
f32 = @formula(distressed ~ temperature)
m31 = glm(f31, flights, Binomial(); wts=six)
m32 = glm(f32, flights, Binomial(); wts=six)
replicate("Model (3.1): temperature and pressure (p. 948)", collect(zip(["alpha", "beta", "gamma", "se alpha", "se beta", "se gamma", "G2, 20 df"],
	[coef(m31); stderror(m31); deviance(m31)], ["2.520", "-.0983", ".00848", "3.487", ".045", ".0077", "16.546"])))
replicate("Model (3.2): temperature (p. 948)", collect(zip(["alpha", "beta", "se alpha", "se beta", "G2, 21 df"],
	[coef(m32); stderror(m32); deviance(m32)], ["5.085", "-.1156", "3.052", ".047", "18.086"])))
increment = deviance(m32) - deviance(m31)
replicate("Dropping pressure (p. 948)", [("increment in G2, 1 df", increment, "1.54")])
@printf("p = %.3f\n", ccdf(Chisq(1), increment))
pressure = ccdf(Chisq(1), increment) <= 0.05  # the branch: keep pressure only if significant at 5%
f, model = pressure ? (f31, m31) : (f32, m32)

## 3. The binary model, and Challenger's 31°F

A flight's six joints share one launch, so their damage may not be independent. The binary model asks only whether a flight had any damage, with probability p*(t) = 1 − (1 − p(t))⁶ if the binomial model holds (p. 949). The two models give close curves of the expected number of damaged O-rings (Fig. 4), which supports the binomial one. At 31°F, Challenger's temperature, both expect about five of the six (pp. 949 and 953).

In [ ]:
binary = glm(@formula(incident ~ temperature), flights, Binomial())
replicate("Binary model (p. 949)", [("alpha*", coef(binary)[1], "15.043"), ("beta*", coef(binary)[2], "-.2322")])
replicate("At 31°F (p. 953)", [("expected damaged O-rings", 6predict(m32, new([31]))[1], "5")])
@printf("binary model: %.2f (published: about four to five of the six, p. 949)
", 6(1 - (1 - predict(binary, new([31]))[1])^(1 / 6)))

grid = 30:85
scatter(flights.temperature, ch.n_failures, label="flights", xlabel="joint temperature, °F", ylabel="expected damaged O-rings", ylims=(0, 6))
plot!(grid, 6predict(model, new(grid)), label="binomial model")
plot!(grid, 6(1 .- (1 .- predict(binary, new(grid))) .^ (1 / 6)), linestyle=:dash, label="binary model")
vline!([31], color=:gray, label="Challenger, 31°F")

## 4. How sure? The parametric bootstrap

Instead of asymptotic standard errors, the paper uses Efron's (1979) parametric bootstrap (p. 949): take the fitted model as true, draw new damage counts for the 23 flights from it, refit, and repeat. The 5% and 95% points of the refitted curves give a 90% interval at each temperature (Fig. 5). The intervals are wide below 65°F, where there are few flights; at 30°F the paper's is "about (1, 6)" (p. 950). A bootstrap is random, so its last digits change with the seed and the number of samples.

In [ ]:
"""
	parametric(m, f, response, B)

Refit the formula `f` to `B` samples of the field joints' damage drawn from the fitted binomial model `m`, with the sample as `response`
(Efron 1979; p. 949). A sample whose fit fails is skipped.
"""
function parametric(m, f, response, B)
	p = predict(m)  # the fitted probability per joint of each flight
	fits = []
	for _ in 1:B
		y = rand.(Binomial.(6, p)) ./ 6
		try
			push!(fits, glm(f, merge(flights, NamedTuple{(response,)}((y,))), Binomial(); wts=six))
		catch
		end
	end
	fits
end

Random.seed!(1989)
boot = parametric(model, f, :distressed, 1000)
curves = reduce(hcat, [6predict(b, new(grid)) for b in boot])
lower, upper = quantile.(eachrow(curves), 0.05), quantile.(eachrow(curves), 0.95)
@printf("%d of 1000 bootstrap fits succeeded\n", length(boot))
replicate("90% bootstrap interval at 30°F (p. 950)", [("5%", lower[1], "1"), ("95%", upper[1], "6")])

plot(grid, 6predict(model, new(grid)), ribbon=(6predict(model, new(grid)) .- lower, upper .- 6predict(model, new(grid))),
	fillalpha=0.25, label="expected, with 90% bootstrap interval", xlabel="joint temperature, °F", ylabel="expected damaged O-rings", ylims=(0, 6))
scatter!(flights.temperature, ch.n_failures, label="flights")

## 5. Which flight matters? Delta beta

Each flight is dropped in turn and the model refitted; the change in the temperature coefficient, divided by the standard deviation of the 23 refitted coefficients, is δᵢ = (β(all) − β(−i)) / S(β) (p. 950). One flight stands out by about four standard deviations: the 21st, 61-A, with two damaged O-rings at 75°F, the point that made the teleconference's plot a U. The other large values are at flights 2, 11 and 14, all with damage.

In [ ]:
dropped(i) = glm(f32, map(column -> column[setdiff(1:23, i)], flights), Binomial(); wts=six[setdiff(1:23, i)])
loo = [coef(dropped(i))[2] for i in 1:23]
delta = (coef(m32)[2] .- loo) ./ std(loo)
@printf("delta beta, flight 21 (%s): %.2f (published: about four standard deviations, p. 950)\n", ch.flight[21], delta[21])
@printf("next largest: %s\n", join(["$i ($(ch.flight[i])): $(round(delta[i], digits=2))" for i in sortperm(abs.(delta), rev=true)[2:4]], ", "))
scatter(1:23, delta, xlabel="flight, in launch order", ylabel="delta beta", label="", series_annotations=text.(ch.flight, 7, :bottom))

## 6. Is the model linear in temperature?

Adding a squared term in the centred temperature tests for curvature (p. 951). The fall in G² is far below 3.84, the 95% point of a chi-square with 1 df, so the linear model stays. The paper also checks linearity with smoothing (Cleveland and Devlin 1988), local deviances (Fowlkes 1987) and ACE (Breiman and Friedman 1985), which print no values to replicate.

In [ ]:
c = flights.temperature .- mean(flights.temperature)
quadratic = glm(@formula(distressed ~ c + c2), merge(flights, (c = c, c2 = c .^ 2)), Binomial(); wts=six)
replicate("Quadratic in centred temperature (p. 951)", [("alpha", coef(quadratic)[1], "-3.1156"), ("beta", coef(quadratic)[2], "-.0748"),
	("gamma", coef(quadratic)[3], ".0041"), ("G2", deviance(quadratic), "17.592"), ("fall in G2, 1 df", deviance(m32) - deviance(quadratic), ".494")])

## 7. The nozzle joints

Each rocket also has two nozzle joints, sealed by the same primary O-ring. A heat incident on a nozzle joint is any erosion or blowby of its primary O-ring (p. 952). The binomial model fitted them poorly, so the paper fits the binary model, with the nozzle's own leak-check pressure (p. 953). Temperature acts as on the field joints, but pressure matters more, and at 100 and 200 psi the nozzle O-rings are far more prone to damage (Fig. 11).

In [ ]:
nozzle = (temperature = flights.temperature, pressure = Float64.(ch.nozzle_pressure),
	incident = Float64.((ch.nozzle_erosion .+ ch.nozzle_blowby) .> 0))  # any primary O-ring erosion or blowby
mn = glm(@formula(incident ~ temperature + pressure), nozzle, Binomial())
replicate("Nozzle joints, binary model (p. 953)", [("alpha", coef(mn)[1], "13.08"), ("beta", coef(mn)[2], "-.238"),
	("gamma, per 10 psi", 10coef(mn)[3], ".35"), ("G2, 20 df", deviance(mn), "18")])
@printf("gamma, per psi: %.4f
", coef(mn)[3])
plot(xlabel="joint temperature, °F", ylabel="P(heat incident on a nozzle joint)", ylims=(0, 1), legend=:bottomleft)
for s in (50, 100, 200)
	plot!(grid, predict(mn, new(grid; pressure=s)), label="$s psi")
end
plot!()

## 8. The probability of disaster

Damage is not failure. A field joint fails completely only after four events, each conditional on the ones before (p. 953): (a) the primary O-ring erodes, (b) it is blown by, (c) the secondary O-ring erodes, and (d) the secondary fails. So pF = pa·pb·pc·pd, and with six field joints failing independently, the probability that at least one fails is pFF = 1 − (1 − pF)⁶.

- pa comes from a binomial model of primary erosion alone, on temperature and pressure: pressure is not significant, but it is kept for prediction (p. 953).
- pb: of the 7 eroded field-joint O-rings, 2 were blown by, and of the 17 nozzle ones, 5; pooled, 7/24 (p. 954).
- pc: of these 2 and 5, one of each had secondary erosion; pooled, 2/7 (p. 954).
- pd: a secondary O-ring had never failed. The engineers judged it no more reliable than the primary, so pd = pb (p. 955).

At 31°F and 200 psi, Challenger's conditions, pFF is .13; at 60°F it would have been .019, a risk about 600% higher on the cold morning (p. 955). Estimating pc from the field joints alone, 1/2, raises them to .218 and .032.

In [ ]:
erosion = glm(@formula(eroded ~ temperature + pressure), flights, Binomial(); wts=six)
replicate("Field-joint primary erosion (p. 953)", [("alpha", coef(erosion)[1], "7.8"), ("beta", coef(erosion)[2], "-.17"), ("gamma", coef(erosion)[3], ".0024")])
pa(t) = predict(erosion, new([t]))[1]  # primary erosion, per field joint
pb = 7 / 24  # blowby, given erosion: field and nozzle joints pooled
pc = 2 / 7  # secondary erosion, given erosion and blowby: pooled
pd = pb  # secondary failure, given all of the above: engineering judgment
pFF(t; pc=pc) = 1 - (1 - pa(t) * pb * pc * pd)^6
replicate("The risk chain (pp. 953-955)", [("pa at 31°F", pa(31), ".95"), ("pb", pb, ".292"), ("pc", pc, ".286"),
	("pF at 31°F", pa(31) * pb * pc * pd, ".023"), ("pFF at 31°F", pFF(31), ".13"), ("pFF at 60°F", pFF(60), ".019"),
	("pFF at 31°F, pc = 1/2", pFF(31; pc=1 / 2), ".218"), ("pFF at 60°F, pc = 1/2", pFF(60; pc=1 / 2), ".032")])
@printf("risk at 31°F against 60°F: %.0f%% higher (published: 600%%)\n", 100(pFF(31) / pFF(60) - 1))

Random.seed!(1986)
bootpa = parametric(erosion, @formula(eroded ~ temperature + pressure), :eroded, 500)
B = reduce(hcat, coef.(bootpa))
println("
90% bootstrap intervals for the erosion model (p. 953)")
for (i, label, published) in ((1, "alpha", "(-.1, 15.7)"), (2, "beta", "(-.28, -.06)"), (3, "gamma", "(-.012, .016)"))
	@printf("%-8s(%.4f, %.4f)   published %s
", label, quantile(B[i, :], 0.05), quantile(B[i, :], 0.95), published)
end
@printf("runaway fits, |gamma| above .05: %d of %d
", count(>(0.05), abs.(B[3, :])), length(bootpa))

## 9. The uncertainty of the risk

pb and pc rest on 24 and 7 O-rings, so the paper gives pFF a posterior distribution by Bayesian bootstrap arguments (Rubin 1981): pa's is its bootstrap distribution, the 500 fits of Section 8; with a uniform prior, pb's is Beta(7 + 1, 24 + 1 − 7) and pc's Beta(2 + 1, 7 + 1 − 2); and pd's is pb's (p. 956). From 500 draws, the posterior of pFF at 31°F and 200 psi has median .1378 and 90% interval (.0275, .3704); at 60°F, median .0205 and interval (.0031, .0764) (p. 957). A beta distribution fitted by maximum likelihood to the 31°F draws has α = 1.598 and β = 8.068 (Fig. 17). These are simulations: the paper's 500 draws cannot be reproduced, so its values are matched to the precision 500 draws allow, about one decimal.

In [ ]:
Random.seed!(28)
posterior(t) = [1 - (1 - predict(b, new([t]))[1] * (q = rand(Beta(8, 18)); q * q) * rand(Beta(3, 6)))^6 for b in bootpa]
p31, p60 = posterior(31), posterior(60)
replicate("Posterior of pFF at 31°F (pp. 956-957)", [("median", median(p31), ".1378"), ("5%", quantile(p31, 0.05), ".0275"), ("95%", quantile(p31, 0.95), ".3704"), ("mean", mean(p31), ".1641")])
replicate("Posterior of pFF at 60°F (p. 957)", [("median", median(p60), ".0205"), ("5%", quantile(p60, 0.05), ".0031"), ("95%", quantile(p60, 0.95), ".0764")])
@printf("mean: %.4f (published: .0048, below half the median, which no distribution on [0, 1] allows)
", mean(p60))
@printf("
draws at 31°F equal to 0, from runaway fits: %d
", sum(p31 .== 0))
beta31 = fit_mle(Beta, filter(>(0), p31))  # a beta likelihood is zero at 0
replicate("Beta fit at 31°F (Fig. 17, p. 956)", [("alpha", beta31.α, "1.598"), ("beta", beta31.β, "8.068")])
@printf("mean of the beta fit: %.3f (published fit: 1.598 / (1.598 + 8.068) = %.3f)
", mean(beta31), 1.598 / (1.598 + 8.068))

histogram(p31, normalize=:pdf, bins=30, fillalpha=0.4, label="31°F", xlabel="probability of a field-joint failure, pFF", ylabel="density")
histogram!(p60, normalize=:pdf, bins=30, fillalpha=0.4, label="60°F")
plot!(0:0.005:0.8, pdf.(beta31, 0:0.005:0.8), label="beta fit, 31°F", xlims=(0, 0.8))

## 10. What differs, and why

Every coefficient, standard error and G² of Models (3.1) and (3.2), the binary and quadratic models, the bootstrap interval at 30°F, the delta beta of flight 61-A and the point estimates of the risk chain agree to the published precision. The rest are listed here, explained where the cause was found.

- **The fall in G² of the quadratic (.495 against .494).** The paper subtracts the rounded values: 18.086 − 17.592 = .494.
- **The nozzle model.** It needs the 13 flights with any nozzle erosion or blowby. Table 1's "erosion or blowby" column lists only 10: it leaves out 51-F, 51-I and 61-A, whose nozzle erosion is 1. With those 10, α = 19.67 and β = −.320. The printed γ = .35 fits a pressure in units of 10 psi, or a misplaced decimal point: per psi it is .035. β is −.2386, printed −.238: truncated rather than rounded.
- **The erosion model's γ (.0026 against .0024).** α, β and pa at 31°F agree. Not explained.
- **pFF with pc = 1/2 (.220 against .218, and .033 against .032).** Scaling the rounded pF = .023 by .5/.286 gives .218, but the same does not give .032 at 60°F. Not explained.
- **The bootstrap intervals of the erosion model.** About a third of the 500 samples have no maximum likelihood estimate: one coefficient runs off to infinity, GLM.jl stops at a huge value, and the intervals take those fits in. The paper's narrower intervals suggest its program stopped earlier or dropped such samples; it doesn't say which. Not replicated.
- **The posterior (Section 9).** It is a simulation, and the paper's 500 draws cannot be reproduced. The medians and lower points agree to two decimals. The upper point at 31°F (.409 against .370) also includes the runaway bootstrap fits above, two of which give pa = 0.
- **The posterior mean at 60°F (.0251 against .0048).** .0048 cannot be right. A probability is never negative, so at least half of its distribution lies above the median, .0205, and the mean is at least .0205 / 2. The abstract repeats .004.
- **The beta fit (1.393 and 6.913 against 1.598 and 8.068).** The paper uses Gnanadesikan, Pinkham and Hughes's (1967) maximum likelihood from the smallest order statistics; here it is full maximum likelihood, on other draws. The fitted means agree: .168 against .165.

**Now in Compose › New Methods.** GLM.jl fits every model here, but has no parametric bootstrap and no delta-beta influence statistics for a GLM. The New Methods notebook **Parametric bootstrap and delta-beta influence for binomial GLMs** writes both out as documented functions (`parametric` and `deltabeta`), tested against this paper's published values. Open it from **Compose › New Methods**.

## References

- Breiman, L. and Friedman, J. H. (1985). Estimating optimal transformations for multiple regression and correlation. *Journal of the American Statistical Association* 80(391), 580-598.
- Cleveland, W. S. and Devlin, S. J. (1988). Locally weighted regression: an approach to regression analysis by local fitting. *Journal of the American Statistical Association* 83(403), 596-610.
- Dalal, S. R., Fowlkes, E. B. and Hoadley, B. (1989). Risk analysis of the space shuttle: pre-Challenger prediction of failure. *Journal of the American Statistical Association* 84(408), 945-957. doi:10.1080/01621459.1989.10478858
- Efron, B. (1979). Bootstrap methods: another look at the jackknife. *The Annals of Statistics* 7(1), 1-26.
- Fowlkes, E. B. (1987). Some diagnostics for binary logistic regression via smoothing. *Biometrika* 74(3), 503-515.
- Gnanadesikan, R., Pinkham, R. S. and Hughes, L. P. (1967). Maximum likelihood estimation of the parameters of the beta distribution from smallest order statistics. *Technometrics* 9(4), 607-620.
- Presidential Commission on the Space Shuttle Challenger Accident (1986). *Report to the President*, Vol. 1, pp. 129-131: the data.
- Rubin, D. B. (1981). The Bayesian bootstrap. *The Annals of Statistics* 9(1), 130-134.